# 01 — FP32 Baselines & Input Bit-Depth Sweep

For each trained checkpoint (`seed_1`, `seed_2`, `seed_42`):

1. **FP32 baseline** — `input_quant_bits=8`
2. **FP32 reduced bit-depth** — `input_quant_bits` ∈ {4, 2, 1}
3. **FP16 full sweep** — `input_quant_bits` ∈ {8, 4, 2, 1}

Results are saved under `runs/val_infer/<seed>/`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [2]:
import torch
import pandas as pd

from src.config import ExperimentConfig, with_overrides
from src.runner import run_experiment
from utils.utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [ ]:
CKPT_ROOT = PROJECT_ROOT / "checkpoints"
INPUT_BITS = [8, 4, 2, 1]

checkpoints = {}
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if seed_dir.is_dir() and (seed_dir / "best.pth").exists():
            checkpoints[(bits, seed_dir.name)] = str(seed_dir / "best.pth")

print("Checkpoints found:")
for (bits, seed), path in checkpoints.items():
    print(f"  {bits}bit / {seed}: {path}")


Checkpoints found:
  seed_1: /home/pf4636/code/resnet/quantized_resnets/training/checkpoints/fp32/seed_1/best.pth
  seed_2: /home/pf4636/code/resnet/quantized_resnets/training/checkpoints/fp32/seed_2/best.pth
  seed_42: /home/pf4636/code/resnet/quantized_resnets/training/checkpoints/fp32/seed_42/best.pth


## FP32 Full Sweep (b=8, 4, 2, 1)

In [ ]:
fp32_records = []

for (bits, seed), ckpt_path in checkpoints.items():
    print(f"\n{'='*60}")
    print(f"  {bits}bit / {seed}  |  precision: fp32")
    print(f"{'='*60}")

    base = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=500,
        output_root=f"../runs/val_infer/fp32_{bits}bit/{seed}",
    )

    cfg = with_overrides(base, model_precision="fp32", input_quant_bits=bits)
    print(f"\n--- fp32 | input_bits={bits} ---")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=ckpt_path)
    fp32_records.append(payload)
    print(f"  top1={payload['results']['top1_acc']:.2f}%  top5={payload['results']['top5_acc']:.2f}%")



  SEED: seed_1  |  precision: fp32

--- fp32 | input_bits=8 ---
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 2.86 ms/batch
  Batch [50/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.47 ms/batch
  Batch [60/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.47 ms/batch
  Batch [70/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.27 ms/batch
  Batch [80/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.23 ms/batch
  Batch [90/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.35 ms/batch
  Batch [100/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.32 ms/batch
  Batch [110/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.28 ms/batch
  Batch [120/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.18 ms/batch
  Batch [130/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.16 ms/batch
  Ba

## FP16 Full Sweep (b=8, 4, 2, 1)

In [ ]:
fp16_records = []

for (bits, seed), ckpt_path in checkpoints.items():
    print(f"\n{'='*60}")
    print(f"  {bits}bit / {seed}  |  precision: fp16")
    print(f"{'='*60}")

    base = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=500,
        output_root=f"../runs/val_infer/fp32_{bits}bit/{seed}",
    )

    cfg = with_overrides(base, model_precision="fp16", input_quant_bits=bits)
    print(f"\n--- fp16 | input_bits={bits} ---")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=ckpt_path)
    fp16_records.append(payload)
    print(f"  top1={payload['results']['top1_acc']:.2f}%  top5={payload['results']['top5_acc']:.2f}%")



  SEED: seed_1  |  precision: fp16

--- fp16 | input_bits=8 ---
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.22 ms/batch
  Batch [50/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 2.41 ms/batch
  Batch [60/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 2.66 ms/batch
  Batch [70/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 2.56 ms/batch
  Batch [80/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 2.63 ms/batch
  Batch [90/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 2.80 ms/batch
  Batch [100/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 2.91 ms/batch
  Batch [110/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 2.96 ms/batch
  Batch [120/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 2.99 ms/batch
  Batch [130/500] Top-1: 100.00% | Top-5: 100.00% | Infer: 3.02 ms/batch
  Ba

## Results Summary

In [ ]:
all_rows = []
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if not seed_dir.is_dir():
            continue
        run_dir = f"../runs/val_infer/fp32_{bits}bit/{seed_dir.name}"
        runs = load_runs(run_dir, status="ok")
        for r in flatten_runs(runs):
            r["seed"] = seed_dir.name
            r["input_bits_trained"] = bits
            all_rows.append(r)

df = pd.DataFrame(all_rows)

display_cols = [
    "seed",
    "input_bits_trained",
    "run_id",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]

df[display_cols].sort_values(
    ["input_bits_trained", "seed", "cfg.model_precision", "cfg.input_quant_bits"],
    ascending=[False, True, True, False],
)


,seed,run_id,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
3,seed_1,resnet18_pytorch_fp16_in8b_cuda_bs1,fp16,8,98.936170,100.000000,2.996082,333.769231,470
2,seed_1,resnet18_pytorch_fp16_in4b_cuda_bs1,fp16,4,97.234043,100.000000,3.025733,330.498440,470
1,seed_1,resnet18_pytorch_fp16_in2b_cuda_bs1,fp16,2,58.936170,82.127660,3.097284,322.863540,470
0,seed_1,resnet18_pytorch_fp16_in1b_cuda_bs1,fp16,1,33.829787,60.638298,3.082136,324.450368,470
7,seed_1,resnet18_pytorch_fp32_in8b_cuda_bs1,fp32,8,98.936170,100.000000,3.151437,317.315589,470
6,seed_1,resnet18_pytorch_fp32_in4b_cuda_bs1,fp32,4,97.234043,100.000000,2.787812,358.704183,470
5,seed_1,resnet18_pytorch_fp32_in2b_cuda_bs1,fp32,2,58.936170,82.127660,3.035384,329.447618,470
4,seed_1,resnet18_pytorch_fp32_in1b_cuda_bs1,fp32,1,33.829787,60.638298,2.801803,356.913054,470
11,seed_2,resnet18_pytorch_fp16_in8b_cuda_bs1,fp16,8,98.085106,100.000000,2.693583,371.252731,470
10,seed_2,resnet18_pytorch_fp16_in4b_cuda_bs1,fp16,4,96.595745,100.000000,2.918227,342.673840,470


In [ ]:
avg_df = df[df["cfg.backend"] == "pytorch"].groupby(
    ["cfg.backend", "cfg.model_precision", "input_bits_trained"]
).agg(
    top1_mean=("res.top1_acc", "mean"),
    top1_std=("res.top1_acc", "std"),
    top5_mean=("res.top5_acc", "mean"),
    top5_std=("res.top5_acc", "std"),
    infer_ms_mean=("res.infer_ms_avg", "mean"),
    infer_ms_std=("res.infer_ms_avg", "std"),
    throughput_mean=("res.throughput_infer_sps", "mean"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["cfg.model_precision", "input_bits_trained"],
    ascending=[True, False],
).reset_index(drop=True)
avg_df


,cfg.backend,cfg.model_precision,cfg.input_quant_bits,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean
0,pytorch,fp16,8,94.468,7.015,99.574,0.737,2.968,0.262,338.696
1,pytorch,fp16,4,93.191,6.457,99.291,1.228,2.910,0.119,343.986
2,pytorch,fp16,2,56.028,3.982,80.355,1.931,2.997,0.101,333.895
3,pytorch,fp16,1,32.411,1.494,58.794,1.919,3.133,0.048,319.222
4,pytorch,fp32,8,94.468,7.015,99.574,0.737,3.089,0.055,323.786
5,pytorch,fp32,4,93.191,6.457,99.291,1.228,2.841,0.084,352.216
6,pytorch,fp32,2,55.957,3.940,80.355,1.931,2.951,0.147,339.456
7,pytorch,fp32,1,32.340,1.489,58.794,1.919,2.840,0.126,352.600


In [ ]:
avg_df[["cfg.model_precision", "input_bits_trained",
        "top1_mean", "top1_std", "top5_mean", "top5_std",
        "infer_ms_mean", "infer_ms_std"]].assign(
    top1=lambda d: d.apply(lambda r: f"{r['top1_mean']:.2f} ± {r['top1_std']:.2f}", axis=1),
    top5=lambda d: d.apply(lambda r: f"{r['top5_mean']:.2f} ± {r['top5_std']:.2f}", axis=1),
    infer_ms=lambda d: d.apply(lambda r: f"{r['infer_ms_mean']:.3f} ± {r['infer_ms_std']:.3f}", axis=1),
)[["cfg.model_precision", "input_bits_trained", "top1", "top5", "infer_ms"]]


,cfg.model_precision,cfg.input_quant_bits,top1,top5,infer_ms
0,fp16,8,94.47 ± 7.01,99.57 ± 0.74,2.968 ± 0.262
1,fp16,4,93.19 ± 6.46,99.29 ± 1.23,2.910 ± 0.119
2,fp16,2,56.03 ± 3.98,80.36 ± 1.93,2.997 ± 0.101
3,fp16,1,32.41 ± 1.49,58.79 ± 1.92,3.133 ± 0.048
4,fp32,8,94.47 ± 7.01,99.57 ± 0.74,3.089 ± 0.055
5,fp32,4,93.19 ± 6.46,99.29 ± 1.23,2.841 ± 0.084
6,fp32,2,55.96 ± 3.94,80.36 ± 1.93,2.951 ± 0.147
7,fp32,1,32.34 ± 1.49,58.79 ± 1.92,2.840 ± 0.126


In [13]:
out_path = PROJECT_ROOT / "results" / "pytorch_avg_results_val.json"
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")

Saved to /home/pf4636/code/resnet/quantized_resnets/results/pytorch_avg_results_val.json
